In [29]:
# ======================================================
# 🐦 BIRD CLASSIFIER (AUDIO + IMAGE USING YOUR DATA)
# ======================================================

import os
import numpy as np
import librosa
import joblib
import torch
from torchvision import datasets, transforms, models
from torch import nn, optim
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# ======================================================
# 🔊 AUDIO SECTION
# ======================================================

def extract_audio_features(file):
    try:
        y, sr = librosa.load(file, sr=22050)
        y = librosa.util.normalize(y)
        y, _ = librosa.effects.trim(y)

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        mel = librosa.feature.melspectrogram(y=y, sr=sr)

        features = np.hstack([
            np.mean(mfcc.T, axis=0),
            np.mean(chroma.T, axis=0),
            np.mean(mel.T, axis=0)
        ])

        return features
    except:
        return None


# 🔹 PATH (FIXED)
audio_dataset_path = r"D:\bird project\datas"

X = []
y = []

print("📂 Loading AUDIO dataset...")

for label in os.listdir(audio_dataset_path):
    folder = os.path.join(audio_dataset_path, label)

    if not os.path.isdir(folder):
        continue

    for file in os.listdir(folder):
        if file.endswith(".wav") or file.endswith(".mp3"):
            file_path = os.path.join(folder, file)

            features = extract_audio_features(file_path)

            if features is not None:
                X.append(features)
                y.append(label)

print(f"✅ Audio samples: {len(X)}")

# 🔤 Encode
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

# Train
print("🤖 Training AUDIO model...")
audio_model = RandomForestClassifier(n_estimators=300, random_state=42)
audio_model.fit(X_train, y_train)

print(f"🎯 Audio Accuracy: {audio_model.score(X_test, y_test)*100:.2f}%")

joblib.dump(audio_model, "bird_audio_model.pkl")
joblib.dump(le, "label_encoder.pkl")


# 🔮 AUDIO PREDICT
def predict_audio(file):
    features = extract_audio_features(file)

    if features is None:
        print("❌ Audio error")
        return

    probs = audio_model.predict_proba([features])[0]
    idx = probs.argmax()

    print("\n🎧 AUDIO RESULT")
    print("Prediction:", le.inverse_transform([idx])[0])
    print("Confidence:", probs[idx]*100)


# ======================================================
# 🖼️ IMAGE SECTION (YOUR DATA)
# ======================================================

image_dataset_path = r"D:\bird project\datas"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

print("\n📂 Loading IMAGE dataset...")

dataset = datasets.ImageFolder(image_dataset_path, transform=transform)
train_loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

print("Classes:", dataset.classes)

# Load pretrained model
image_model = models.resnet18(pretrained=True)

# Modify final layer
num_classes = len(dataset.classes)
image_model.fc = nn.Linear(image_model.fc.in_features, num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(image_model.parameters(), lr=0.001)

print("🤖 Training IMAGE model...")

for epoch in range(3):   # increase to 5–10 for better accuracy
    total_loss = 0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = image_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

torch.save(image_model.state_dict(), "bird_image_model.pth")

print("💾 Image model saved!")


# 🔮 IMAGE PREDICT
def predict_image(img_path):
    img = Image.open(img_path).convert("RGB")
    img = transform(img).unsqueeze(0)

    image_model.eval()
    with torch.no_grad():
        output = image_model(img)

    pred = torch.argmax(output, 1).item()

    print("\n🖼️ IMAGE RESULT")
    print("Prediction:", dataset.classes[pred])




📂 Loading AUDIO dataset...
✅ Audio samples: 347
🤖 Training AUDIO model...
🎯 Audio Accuracy: 92.86%

📂 Loading IMAGE dataset...
Classes: ['Sparrow', 'Spotted Dove', 'crow', 'peacock', 'pigeon']
🤖 Training IMAGE model...


C:\Users\gokil\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\gokil\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\gokil\anaconda3\Lib\site-packages\PIL\Image.py:1000: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1, Loss: 61.5146
Epoch 2, Loss: 23.6508
Epoch 3, Loss: 15.6299
💾 Image model saved!


In [33]:
# ==============================
# 🧪 TEST BOTH
# ==============================

audio_file = r"D:/bird project/datas/peacock/XC891629 - Grey Peacock-Pheasant - Polyplectron bicalcaratum.wav"
image_file = r"D:\bird project\datas\Spotted Dove\1,100+ Spotted Dove Stock Photos.jpg"

if os.path.exists(test_file):
    print("\n🎧 Testing with:", test_file)
    predict_bird(test_file)
else:
    print("\n⚠️ Put a test file (test.mp3 or test.wav) in this folder")
if os.path.exists(audio_file):
    predict_audio(audio_file)
else:
    print("⚠️ Audio file missing")

if os.path.exists(image_file):
    predict_image(image_file)
else:
    print("⚠️ Image file missing")


🎧 Testing with: D:/bird project/datas/Spotted Dove/XC1035372 - Spotted Dove - Spilopelia chinensis chinensis.wav

🔎 Top Predictions:
Spotted Dove : 98.67%
peacock : 1.00%
crow : 0.33%
✅ Final Prediction: Spotted Dove

🎧 AUDIO RESULT
Prediction: peacock
Confidence: 66.33333333333333

🖼️ IMAGE RESULT
Prediction: Spotted Dove


In [79]:
joblib.dump(dataset.classes, "classes.pkl")

['classes.pkl']

In [86]:
!pip install onnx
%pip install onnx

Note: you may need to restart the kernel to use updated packages.


In [88]:
!pip install onnxruntime

   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.1/12.9 MB 550.5 kB/s eta 0:00:24
    --------------------------------------- 0.2/12.9 MB 1.5 MB/s eta 0:00:09
   - -------------------------------------- 0.4/12.9 MB 2.0 MB/s eta 0:00:07
   - -------------------------------------- 0.6/12.9 MB 2.6 MB/s eta 0:00:05
   -- ------------------------------------- 0.9/12.9 MB 3.3 MB/s eta 0:00:04
   --- ------------------------------------ 1.2/12.9 MB 3.6 MB/s eta 0:00:04
   --- ------------------------------------ 1.3/12.9 MB 3.3 MB/s eta 0:00:04
   ---- ----------------------------------- 1.5/12.9 MB 3.6 MB/s eta 0:00:04
   ----- -----------------------

In [116]:
!pip install onnxconverter-common

   ---------------------------------------- 0.0/89.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/89.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/89.5 kB ? eta -:--:--
   ---- ----------------------------------- 10.2/89.5 kB ? eta -:--:--
   ------------- -------------------------- 30.7/89.5 kB 435.7 kB/s eta 0:00:01
   --------------------------- ------------ 61.4/89.5 kB 544.7 kB/s eta 0:00:01
   ---------------------------------------- 89.5/89.5 kB 634.2 kB/s eta 0:00:00


In [112]:
import zipfile
import os

with zipfile.ZipFile("bird_image_model.zip", "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write("bird_image_model.onnx")

print("✅ ONNX compressed into ZIP")
from IPython.display import FileLink

FileLink("bird_image_model.zip")

✅ ONNX compressed into ZIP


C:\Users\gokil\bird_image_model.zip

In [37]:
from IPython.display import FileLink

FileLink("bird_model.pkl")


C:\Users\gokil\bird_model.pkl

In [100]:
import joblib

joblib.dump(dataset.classes, "classes.pkl")

print("classes.pkl saved")

from IPython.display import FileLink

FileLink("classes.pkl")

classes.pkl saved


C:\Users\gokil\classes.pkl

In [39]:
FileLink("label_encoder.pkl")

C:\Users\gokil\label_encoder.pkl